In [1]:
import numpy as np

# 1. Configuration & Constants
W, C1, C2 = 0.5, 1.5, 1.5
NUM_PARTICLES = 15
ITERATIONS = 30

# Bounds for [Kp, Ki, Kd] network controller gains
MIN_LIMITS = np.array([0.1, 0.01, 0.01])
MAX_LIMITS = np.array([10.0, 2.0, 5.0])

# 2. Mock Network Simulation (The Fitness Function)
def evaluate_network_penalty(x):
    Kp, Ki, Kd = x
    # Simulates network behavior: higher/balanced gains reduce loss and delay
    packet_loss_rate = max(0, 10.0 - (2.0 * Kp + 0.5 * Kd))
    queue_delay_ms = max(5, 50.0 - (5.0 * Kp + 10.0 * Ki))
    queue_error = abs(Kp - 4.5)  # Arbitrary target tracking metric

    # Total Penalty (Minimize this value)
    return (5.0 * packet_loss_rate) + (0.2 * queue_delay_ms) + (10.0 * queue_error)

# 3. Particle Initialization
positions = np.random.uniform(MIN_LIMITS, MAX_LIMITS, (NUM_PARTICLES, 3))
velocities = np.random.uniform(-1, 1, (NUM_PARTICLES, 3))

pbests = np.copy(positions)
pbest_vals = np.array([evaluate_network_penalty(p) for p in pbests])

gbest_idx = np.argmin(pbest_vals)
gbest = np.copy(pbests[gbest_idx])
gbest_val = pbest_vals[gbest_idx]

# 4. Optimization Loop
for t in range(ITERATIONS):
    for i in range(NUM_PARTICLES):
        r1 = np.random.rand(3)
        r2 = np.random.rand(3)

        # Update Velocity & Position
        velocities[i] = (W * velocities[i]) + (C1 * r1 * (pbests[i] - positions[i])) + (C2 * r2 * (gbest - positions[i]))
        positions[i] = np.clip(positions[i] + velocities[i], MIN_LIMITS, MAX_LIMITS)

        # Evaluate Penalty
        fitness = evaluate_network_penalty(positions[i])

        # Update Personal Best
        if fitness < pbest_vals[i]:
            pbests[i] = positions[i]
            pbest_vals[i] = fitness

        # Update Global Best
        if fitness < gbest_val:
            gbest = np.copy(positions[i])
            gbest_val = fitness

# 5. Output Optimal Parameters
print(f"Optimal [Kp, Ki, Kd] Configuration: {gbest}")
print(f"Minimized Network Penalty: {gbest_val:.4f}")


Optimal [Kp, Ki, Kd] Configuration: [4.50001295 2.         4.5138756 ]
Minimized Network Penalty: 1.5001
